In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR1TOT_C.xpt"
csv_file = DATA_DIR / "DR1TOT_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Untouched copy for later fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 DR1TOT_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. KNOWN PANDAS/XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny-value occurrences:",
    f"{int(tiny_counts.sum()):,}"
)

print(
    "Variables containing tiny value:",
    int((tiny_counts > 0).sum())
)


if tiny_counts.sum() > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ].sort_values(
            ascending=False
        )
    )


# ============================================================
# 4. CDC-VALIDATED ZERO VARIABLES
# ============================================================

#
# These are the variables in THIS attached DR1TOT_C XPT
# where pandas produced the tiny zero representation.
#
# CDC allows zero in these fields.
#

ZERO_ALLOWED_COLUMNS = [

    "DR1TNUMF",

    "DR1TKCAL",
    "DR1TPROT",
    "DR1TCARB",
    "DR1TSUGR",
    "DR1TFIBE",
    "DR1TTFAT",
    "DR1TSFAT",
    "DR1TMFAT",
    "DR1TPFAT",
    "DR1TCHOL",
    "DR1TATOC",
    "DR1TATOA",
    "DR1TRET",
    "DR1TVARA",
    "DR1TACAR",
    "DR1TBCAR",
    "DR1TCRYP",
    "DR1TLYCO",
    "DR1TLZ",
    "DR1TVB1",
    "DR1TVB2",
    "DR1TNIAC",
    "DR1TVB6",
    "DR1TFOLA",
    "DR1TFA",
    "DR1TFF",
    "DR1TFDFE",
    "DR1TVB12",
    "DR1TB12A",
    "DR1TVC",
    "DR1TVK",
    "DR1TCALC",
    "DR1TPHOS",
    "DR1TMAGN",
    "DR1TIRON",
    "DR1TZINC",
    "DR1TCOPP",
    "DR1TSODI",
    "DR1TPOTA",
    "DR1TSELE",
    "DR1TCAFF",
    "DR1TTHEO",
    "DR1TALCO",
    "DR1TMOIS",

    "DR1TS040",
    "DR1TS060",
    "DR1TS080",
    "DR1TS100",
    "DR1TS120",
    "DR1TS140",
    "DR1TS160",
    "DR1TS180",

    "DR1TM161",
    "DR1TM181",
    "DR1TM201",
    "DR1TM221",

    "DR1TP182",
    "DR1TP183",
    "DR1TP184",
    "DR1TP204",
    "DR1TP205",
    "DR1TP225",
    "DR1TP226",

    "DR1_320",
    "DR1_330",
    "DR1BWATR",
    "DR1CWATR"
]


# ============================================================
# 5. SAFETY CHECK
# ============================================================

#
# Do not blindly replace a tiny value if it appears
# somewhere we have NOT approved.
#

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_zero_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected_tiny = (
    actual_tiny_columns
    -
    approved_zero_columns
)


if unexpected_tiny:

    raise ValueError(
        "STOP: tiny values were found in unexpected "
        "variables:\n"
        + str(sorted(unexpected_tiny))
    )


print(
    "\nPASS: All tiny values occur only in "
    "validated zero-permitted variables."
)


# ============================================================
# 6. RESTORE TRUE ZERO
# ============================================================

correction_report = []


for col in ZERO_ALLOWED_COLUMNS:

    if col not in df.columns:
        continue

    mask = df[col].eq(
        TINY_VALUE
    )

    count = int(
        mask.sum()
    )

    if count > 0:

        correction_report.append(
            {
                "variable": col,
                "restored_zeros": count
            }
        )

        df.loc[
            mask,
            col
        ] = 0.0


print("\n--- ZERO RESTORATION ---")

correction_df = pd.DataFrame(
    correction_report
)

print(
    correction_df
    .sort_values(
        "restored_zeros",
        ascending=False
    )
    .to_string(index=False)
)


print(
    "\nTotal restored zeros:",
    f"{correction_df['restored_zeros'].sum():,}"
)


# ============================================================
# 7. CLEAN SEQN
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 8. DIETARY MEASUREMENTS TO PRESERVE AS-IS
# ============================================================

#
# These values MUST NOT be rounded or recalculated.
#
# Includes:
#   dietary weights
#   energy/nutrients
#   fatty acids
#   water amounts
#

columns = df.columns.tolist()

nutrient_start = columns.index(
    "DR1TKCAL"
)

nutrient_end = columns.index(
    "DR1TP226"
)


DIETARY_MEASUREMENT_COLUMNS = [

    "WTDRD1",
    "WTDR2D",

    *columns[
        nutrient_start:
        nutrient_end + 1
    ],

    "DR1_320",
    "DR1_330",
    "DR1BWATR",
    "DR1CWATR"
]


# ============================================================
# 9. VERIFY DIETARY MEASUREMENTS WERE NOT ALTERED
#    EXCEPT VALIDATED ZERO RESTORATION
# ============================================================

expected_df = (
    source_df.copy()
)


for col in ZERO_ALLOWED_COLUMNS:

    if col in expected_df.columns:

        expected_df.loc[
            expected_df[col]
            .eq(TINY_VALUE),
            col
        ] = 0.0


print("\n--- DIETARY MEASUREMENT FIDELITY ---")


for col in DIETARY_MEASUREMENT_COLUMNS:

    before = (
        expected_df[col]
        .to_numpy()
    )

    after = (
        df[col]
        .to_numpy()
    )

    same = np.allclose(
        before,
        after,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} was changed."
        )


# ============================================================
# 10. CODE / COUNT VARIABLES
# ============================================================

#
# Everything other than:
#
#   SEQN
#   dietary measurement fields
#
# is a questionnaire/code/count variable in DR1TOT_C.
#
# We first VERIFY that values are whole numbers.
# Only then do we remove unnecessary .0 formatting.
#

CODE_COLUMNS = [

    col
    for col in df.columns

    if col != "SEQN"

    and col
    not in DIETARY_MEASUREMENT_COLUMNS
]


# DR1TNUMF is a count rather than a decimal measurement
CODE_COLUMNS.append(
    "DR1TNUMF"
)


CODE_COLUMNS = list(
    dict.fromkeys(
        CODE_COLUMNS
    )
)


print("\n--- CODE / COUNT FIDELITY ---")


for col in CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values)
    )


    if fractional.any():

        raise ValueError(
            f"{col} contains legitimate fractional values. "
            f"It will NOT be converted to integer."
        )


print(
    "PASS: All code/count variables contain "
    "whole-number source values."
)


# ============================================================
# 11. FORMAT CODES/COUNTS WITHOUT .0
# ============================================================

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


# ============================================================
# 12. CDC BASIC STRUCTURE VALIDATION
# ============================================================

print("\n--- CDC STRUCTURE VALIDATION ---")


if len(df) == 9643:

    print(
        "PASS: Row count = 9,643"
    )

else:

    print(
        "FAIL: Unexpected row count:",
        len(df)
    )


if len(df.columns) == 160:

    print(
        "PASS: Column count = 160"
    )

else:

    print(
        "FAIL: Column count =",
        len(df.columns)
    )


# ============================================================
# 13. CDC DIETARY RECALL STATUS
# ============================================================

EXPECTED_DR1DRSTZ = {

    1: 8894,
    2: 126,
    4: 140,
    5: 483
}


actual_status = (
    df["DR1DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- DR1DRSTZ ---")

print(
    actual_status
)


if actual_status == EXPECTED_DR1DRSTZ:

    print(
        "PASS: DR1DRSTZ matches CDC."
    )

else:

    print(
        "FAIL: DR1DRSTZ differs from CDC."
    )


# ============================================================
# 14. CDC WEIGHT CHECK
# ============================================================

print("\n--- DIETARY WEIGHTS ---")


# WTDRD1
w = df["WTDRD1"]

if (
    int(w.notna().sum()) == 9034
    and
    int(w.isna().sum()) == 609
):

    print(
        "PASS: WTDRD1 valid/missing counts match CDC."
    )

else:

    print(
        "FAIL: WTDRD1 counts differ."
    )


print(
    "WTDRD1 min:",
    w.min()
)

print(
    "WTDRD1 max:",
    w.max()
)


# WTDR2D
w = df["WTDR2D"]

if (
    int(w.notna().sum()) == 8354
    and
    int(w.isna().sum()) == 1289
):

    print(
        "PASS: WTDR2D valid/missing counts match CDC."
    )

else:

    print(
        "FAIL: WTDR2D counts differ."
    )


print(
    "WTDR2D min:",
    w.min()
)

print(
    "WTDR2D max:",
    w.max()
)


# ============================================================
# 15. SELECTED CDC NUTRIENT CHECKS
# ============================================================

print("\n--- SELECTED CDC NUTRIENT CHECKS ---")


# DR1TNUMF
s = df["DR1TNUMF"]

if (
    int(s.notna().sum()) == 9034
    and
    int(s.isna().sum()) == 609
    and
    s.min() == 0
    and
    s.max() == 42
):

    print(
        "PASS: DR1TNUMF matches CDC range/count."
    )

else:

    print(
        "FAIL: DR1TNUMF differs."
    )


# DR1TKCAL
s = df["DR1TKCAL"]

if (
    int(s.notna().sum()) == 8894
    and
    int(s.isna().sum()) == 749
    and
    s.min() == 0
    and
    s.max() == 9724
):

    print(
        "PASS: DR1TKCAL matches CDC range/count."
    )

else:

    print(
        "FAIL: DR1TKCAL differs."
    )


# DR1CWATR
s = df["DR1CWATR"]

if (
    int(s.notna().sum()) == 9160
    and
    int(s.isna().sum()) == 483
    and
    np.isclose(
        s.min(),
        0
    )
    and
    np.isclose(
        s.max(),
        3788.8
    )
):

    print(
        "PASS: DR1CWATR matches CDC range/count."
    )

else:

    print(
        "FAIL: DR1CWATR differs."
    )


# ============================================================
# 16. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact remains."
    )

else:

    raise ValueError(
        "FAIL: tiny values remain."
    )


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# No float_format is supplied.
#
# We do NOT round dietary measurements.
# We do NOT recalculate nutrients.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 18. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. EXACT NUMERIC FIDELITY AFTER CSV EXPORT
# ============================================================

print("\n--- XPT -> CSV DIETARY FIDELITY ---")


for col in DIETARY_MEASUREMENT_COLUMNS:

    expected = (
        expected_df[col]
        .to_numpy()
    )

    exported = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected,
        exported,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        print(
            f"FAIL: {col} changed during export."
        )


# ============================================================
# 20. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    source_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 21. RAW SEQN FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    f.readline()

    bad_seqn = 0

    for line in f:

        seqn_text = (
            line.split(",", 1)[0]
        )

        if seqn_text.endswith(".0"):

            bad_seqn += 1


print(
    "SEQN values ending in .0:",
    bad_seqn
)


if bad_seqn == 0:

    print(
        "PASS: SEQN exported without .0."
    )


print("\n" + "=" * 75)
print("DR1TOT_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 DR1TOT_C XPT -> CSV
Rows: 9,643
Columns: 160

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny-value occurrences: 73,966
Variables containing tiny value: 68

Tiny values by variable:
DR1CWATR    8970
DR1TATOA    8056
DR1TALCO    7675
DR1TP184    6384
DR1TB12A    5895
            ... 
DR1TSODI       1
DR1TPOTA       1
DR1TSELE       1
DR1TCOPP       1
DR1TZINC       1
Length: 68, dtype: int64

PASS: All tiny values occur only in validated zero-permitted variables.

--- ZERO RESTORATION ---
variable  restored_zeros
DR1CWATR            8970
DR1TATOA            8056
DR1TALCO            7675
DR1TP184            6384
DR1TB12A            5895
DR1BWATR            5878
DR1TP225            4768
DR1TTHEO            4545
 DR1_330            3497
DR1TP205            3050
DR1TLYCO            2326
DR1TP226            2319
DR1TCAFF            2213
DR1TM221            2018
 DR1_320            1406
DR1TACAR            1091
DR1TCRYP            1029
DR1TS040             583
DR1TS060  